# Chapter 4 — Poisson Processes

This notebook is a step-by-step tutorial for the concepts in Chapter 4: **Poisson processes**.

The main idea is simple:

> A Poisson process is the continuous-time analog of Bernoulli counting: it counts arrivals over time, has independent increments, and in the stationary case the distribution of an increment depends only on the interval length.

We will build the chapter in layers:

1. Arrival counting processes and the Poisson distribution of counts.
2. Arrival times and exponential interarrival times.
3. Forward recurrence times.
4. Superposition of Poisson processes.
5. Decomposition / thinning of Poisson processes.
6. Compound Poisson processes.
7. Non-stationary Poisson processes.

The notebook includes the chapter's examples, proof ideas, and simulations.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(12345)

# NumPy 2 removed np.trapz; use np.trapezoid when needed.
def poisson_pmf(k, mean):
    if k < 0:
        return 0.0
    return math.exp(-mean + k * math.log(mean) - math.lgamma(k + 1)) if mean > 0 else (1.0 if k == 0 else 0.0)

def poisson_cdf(k, mean):
    return sum(poisson_pmf(j, mean) for j in range(k + 1))

def exp_pdf(t, lam):
    t = np.asarray(t)
    return lam * np.exp(-lam * t) * (t >= 0)

def poisson_path(rate=2.0, T=5.0, rng=rng):
    arrivals = []
    t = 0.0
    while True:
        t += rng.exponential(1 / rate)
        if t > T:
            break
        arrivals.append(t)
    return np.array(arrivals)

def plot_counting_path(arrivals, T=None, title="Counting process sample path"):
    if T is None:
        T = max(arrivals[-1] + 0.5, 1.0) if len(arrivals) else 1.0
    xs = [0.0]
    ys = [0]
    count = 0
    for a in arrivals:
        xs.extend([a, a])
        ys.extend([count, count + 1])
        count += 1
    xs.append(T)
    ys.append(count)
    plt.figure(figsize=(8, 3.5))
    plt.step(xs, ys, where="post")
    plt.scatter(arrivals, np.arange(1, len(arrivals)+1), zorder=3)
    plt.xlabel("t")
    plt.ylabel(r"$N_t$")
    plt.title(title)
    plt.ylim(-0.1, max(1, count) + 0.8)
    plt.grid(True, alpha=0.3)
    plt.show()

## 1. Arrival counting processes

An **arrival process** is a stochastic process $N = \{N_t:t\ge 0\}$ whose sample path counts arrivals up to time $t$.

For almost every outcome $\omega$:

- $t \mapsto N_t(\omega)$ is non-decreasing.
- It is right-continuous.
- $N_0(\omega)=0$.
- It increases only by jumps.

A sample path is a step function. Each jump marks an arrival time.

In [ ]:
arrivals = np.array([0.4, 1.2, 1.7, 2.6, 3.0, 4.7])
plot_counting_path(arrivals, T=5.0, title="A possible arrival counting path")

## 2. Definition of a stationary Poisson process

An arrival process $N = \{N_t:t\ge 0\}$ is a **Poisson process** if:

1. **Unit jumps:** each jump has size $1$.
2. **Stationary increments:** for $s,t\ge 0$, the distribution of $N_{t+s}-N_t$ depends only on $s$.
3. **Independent increments:** $N_{t+s}-N_t$ is independent of the past history up to time $t$.

The parameter $\lambda$ will be the **arrival rate**:

$$
E[N_1]=\lambda.
$$

Thinking model:

- Bernoulli process: many discrete trials, each success counted.
- Poisson process: trials are so fine-grained that time becomes continuous.
- The count over an interval of length $t$ has mean $\lambda t$.

## 3. Deriving the count distribution

Let

$$
f(t)=P(N_t=0).
$$

Using stationarity and independent increments,

$$
P(N_{t+s}=0)=P(N_t=0)P(N_{t+s}-N_t=0)=f(t)f(s).
$$

A non-trivial right-continuous solution of this multiplicative equation has the form

$$
f(t)=e^{-\lambda t}
$$

for some $\lambda\ge 0$.

So the probability of no arrivals in length $t$ is exponential.

### Small interval facts

For very small $t$:

$$
P(N_t\ge 2)=o(t),
$$

and

$$
P(N_t=1)\sim \lambda t.
$$

Interpretation: in a tiny interval, either nothing happens, or one arrival happens. Two or more arrivals are negligible compared with the interval length.

### Theorem: $N_t$ is Poisson distributed

For a Poisson process with rate $\lambda$,

$$
P(N_t=k)=e^{-\lambda t}\frac{(\lambda t)^k}{k!},\qquad k=0,1,2,\ldots
$$

Thus

$$
N_t\sim \operatorname{Poisson}(\lambda t).
$$

### Proof idea using the generating function

Let

$$
G_t(\alpha)=E[\alpha^{N_t}],\qquad 0\le \alpha\le 1.
$$

By independent and stationary increments,

$$
G_{t+s}(\alpha)=G_t(\alpha)G_s(\alpha).
$$

The infinitesimal behavior is:

$$
G_{h}(\alpha)=P(N_h=0)+\alpha P(N_h=1)+O(P(N_h\ge2))
       =1-\lambda h+\alpha\lambda h+o(h).
$$

So

$$
\frac{G_{t+h}(\alpha)-G_t(\alpha)}{h}
  = \lambda(\alpha-1)G_t(\alpha).
$$

Solving this differential equation with $G_0(\alpha)=1$ gives

$$
G_t(\alpha)=e^{-\lambda t(1-\alpha)}.
$$

But

$$
e^{-\lambda t(1-\alpha)}=e^{-\lambda t}\sum_{k=0}^\infty \frac{(\lambda t)^k}{k!}\alpha^k.
$$

Matching coefficients of $\alpha^k$ gives the Poisson formula.

In [ ]:
lam = 3.0
T = 2.0
mean = lam * T
ks = np.arange(0, 16)
pmf = [poisson_pmf(k, mean) for k in ks]

plt.figure(figsize=(8, 3.5))
plt.bar(ks, pmf)
plt.xlabel("k")
plt.ylabel(r"$P(N_t=k)$")
plt.title(rf"Poisson count distribution, $\lambda t = {mean}$")
plt.grid(True, axis="y", alpha=0.3)
plt.show()

## 4. Distribution of increments

For $s,t\ge0$,

$$
N_{t+s}-N_t\sim \operatorname{Poisson}(\lambda s).
$$

More generally, for $0\le t_0<t_1<\cdots<t_n$,

$$
N_{t_1}-N_{t_0},\; N_{t_2}-N_{t_1},\;\ldots,\;N_{t_n}-N_{t_{n-1}}
$$

are independent, and

$$
N_{t_i}-N_{t_{i-1}}\sim \operatorname{Poisson}(\lambda(t_i-t_{i-1})).
$$

This is the main computational engine of the chapter.

### Example 1.16 — joint probability from independent increments

Suppose $N$ is a Poisson process with rate $\lambda=8$. Compute

$$
P(N_{1.2}=17,\; N_{3.7}=22,\; N_{4.3}=36).
$$

Rewrite the event in terms of increments:

$$
N_{1.2}=17,
$$

$$
N_{3.7}-N_{1.2}=5,
$$

$$
N_{4.3}-N_{3.7}=14.
$$

These increments are independent, with means

$$
8(1.2)=9.6,
$$

$$
8(3.7-1.2)=20,
$$

$$
8(4.3-3.7)=4.8.
$$

Therefore

$$
P=e^{-9.6}\frac{9.6^{17}}{17!}
  \cdot e^{-20}\frac{20^5}{5!}
  \cdot e^{-4.8}\frac{4.8^{14}}{14!}.
$$

In [ ]:
p = poisson_pmf(17, 9.6) * poisson_pmf(5, 20.0) * poisson_pmf(14, 4.8)
p

## 5. A simpler characterization

The chapter shows that, for many purposes, it is enough to check:

1. unit jumps, and
2. the expected increment condition

$$
E[N_{t+s}-N_t]=\lambda s.
$$

Together with the qualitative arrival-process assumptions, this characterizes a Poisson process.

Thinking model:

- The full definition says: stationary + independent increments.
- The characterization says: if the process is already qualitatively an arrival process with unit jumps, then the correct linear mean behavior forces the Poisson structure.

## 6. Counts over arbitrary disjoint sets

For a set $B\subseteq [0,\infty)$, let $N_B$ be the number of arrivals whose times lie in $B$.

If $B$ has total length $b$, then

$$
P(N_B=k)=e^{-\lambda b}\frac{(\lambda b)^k}{k!}.
$$

If $B_1,\ldots,B_n$ are disjoint, then

$$
N_{B_1},\ldots,N_{B_n}
$$

are independent.

This extends “interval increments” to arbitrary disjoint unions of intervals.

## 7. Conditional allocation: multinomial splitting inside an interval

Suppose $A_1,\ldots,A_n$ are disjoint intervals whose union is $B$, with lengths $a_1,
\ldots,a_n$, and total length

$$
b=a_1+\cdots+a_n.
$$

Condition on knowing that $N_B=k$. Then the allocation of those $k$ arrivals among the subintervals is multinomial:

$$
P(N_{A_1}=k_1,\ldots,N_{A_n}=k_n\mid N_B=k)
=\frac{k!}{k_1!\cdots k_n!}\left(\frac{a_1}{b}\right)^{k_1}\cdots\left(\frac{a_n}{b}\right)^{k_n},
$$

where $k_1+\cdots+k_n=k$.

Thinking model:

> Given that $k$ arrivals happened somewhere in $B$, their locations behave like $k$ independent uniform picks over $B$.

In [ ]:
# Simulate conditional allocation in an interval split into three parts.
# Given k=10 arrivals in total, lengths 1, 2, 3 imply probabilities 1/6, 2/6, 3/6.
k = 10
probs = np.array([1, 2, 3]) / 6
samples = rng.multinomial(k, probs, size=5000)
samples[:5]

## 8. Long-run behavior of counts

Because counts over equal-size intervals are independent and identically distributed, laws of large numbers apply.

For a Poisson process with rate $\lambda$,

$$
\frac{N_t}{t}\to \lambda
$$

almost surely as $t\to\infty$.

There is also a central-limit approximation:

$$
\frac{N_t-\lambda t}{\sqrt{\lambda t}} \approx N(0,1)
$$

for large $t$.

In [ ]:
lam = 4.0
T = 200.0
arr = poisson_path(lam, T, rng)

ts = np.linspace(1, T, 400)
counts = np.searchsorted(arr, ts, side="right")

plt.figure(figsize=(8, 3.5))
plt.plot(ts, counts / ts, label=r"$N_t/t$")
plt.axhline(lam, linestyle="--", label=r"$\lambda$")
plt.xlabel("t")
plt.ylabel("empirical rate")
plt.title("Long-run empirical rate of a Poisson process")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Part II — Times of arrivals

Let

$$
T_n=\text{time of the }n\text{th arrival}.
$$

Then

$$
\{T_n\le t\}=\{N_t\ge n\}.
$$

The counting process and the arrival-time process contain the same information, just represented differently.

## 9. Exponential interarrival times

Define interarrival times

$$
X_1=T_1,
$$

$$
X_2=T_2-T_1,
$$

and in general

$$
X_n=T_n-T_{n-1}.
$$

For a Poisson process with rate $\lambda$,

$$
P(X_n>t)=e^{-\lambda t},\qquad t\ge0.
$$

So

$$
X_n\sim \operatorname{Exponential}(\lambda),
$$

and the $X_n$ are independent and identically distributed.

The density is

$$
f_X(t)=\lambda e^{-\lambda t},\qquad t\ge0.
$$

Mean and variance:

$$
E[X_n]=\frac1\lambda,
$$

$$
\operatorname{Var}(X_n)=\frac1{\lambda^2}.
$$

In [ ]:
lam = 1.5
t = np.linspace(0, 5, 400)
plt.figure(figsize=(8, 3.5))
plt.plot(t, exp_pdf(t, lam))
plt.xlabel("t")
plt.ylabel("density")
plt.title(rf"Exponential interarrival density, $\lambda={lam}$")
plt.grid(True, alpha=0.3)
plt.show()

## 10. Memorylessness

For an exponential random variable $X$,

$$
P(X>t+s\mid X>t)=P(X>s).
$$

Proof:

$$
P(X>t+s\mid X>t)=\frac{P(X>t+s)}{P(X>t)}
=\frac{e^{-\lambda(t+s)}}{e^{-\lambda t}}
=e^{-\lambda s}.
$$

This is the reason the Poisson process “restarts” after each arrival.

## 11. Characterization by exponential interarrival times

The chapter proves the converse:

> If $T_1,T_2-T_1,T_3-T_2,\ldots$ are independent exponential random variables with common parameter $\lambda$, then the counting process formed from these arrival times is a Poisson process with rate $\lambda$.

This gives a practical way to simulate a Poisson process:

1. Generate iid exponential interarrival times.
2. Cumulatively sum them to get arrival times.
3. Count how many have occurred by time $t$.

In [ ]:
lam = 2.0
T = 6.0
arrivals = poisson_path(lam, T, rng)
plot_counting_path(arrivals, T=T, title="Simulated Poisson process from exponential interarrival times")
arrivals

## 12. Erlang / gamma distribution of $T_n$

Since

$$
T_n=X_1+\cdots+X_n
$$

is a sum of $n$ iid exponential variables with rate $\lambda$, it has the **Erlang distribution**:

$$
P(T_n\le t)=1-\sum_{k=0}^{n-1}e^{-\lambda t}\frac{(\lambda t)^k}{k!}.
$$

Its density is

$$
f_{T_n}(t)=\frac{\lambda(\lambda t)^{n-1}e^{-\lambda t}}{(n-1)!},\qquad t\ge0.
$$

Mean and variance:

$$
E[T_n]=\frac{n}{\lambda},
$$

$$
\operatorname{Var}(T_n)=\frac{n}{\lambda^2}.
$$

In [ ]:
def erlang_pdf(t, n, lam):
    t = np.asarray(t)
    return (lam * (lam*t)**(n-1) * np.exp(-lam*t) / math.factorial(n-1)) * (t >= 0)

t = np.linspace(0, 10, 500)
lam = 1.0
plt.figure(figsize=(8, 3.5))
for n in [1, 2, 5, 10]:
    plt.plot(t, erlang_pdf(t, n, lam), label=rf"$n={n}$")
plt.xlabel("t")
plt.ylabel("density")
plt.title("Erlang densities for arrival times")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Example 2.11 — lifetime of equipment with three components

Suppose a piece of equipment contains three components. Each component lifetime is exponential with rate

$$
\lambda=0.0002 \quad \text{per hour}.
$$

The equipment stops after the third component fails. Therefore, its lifetime is

$$
T_3=X_1+X_2+X_3,
$$

where the $X_i$ are iid exponential with rate $\lambda$.

Thus

$$
E[T_3]=\frac{3}{\lambda}=15000\text{ hours},
$$

and

$$
\operatorname{Var}(T_3)=\frac{3}{\lambda^2}=75\times 10^6\text{ hours}^2.
$$

In [ ]:
lam = 0.0002
E_T3 = 3 / lam
Var_T3 = 3 / lam**2
E_T3, Var_T3

### Example 2.12 — discounted replacement costs

Suppose failures occur according to a Poisson process with rate $\lambda$, and each replacement costs $\beta$ dollars. A cost paid at time $t$ is discounted by $e^{-\alpha t}$.

The total discounted cost is

$$
C=\sum_{n=1}^{\infty}\beta e^{-\alpha T_n}.
$$

Because $T_n=X_1+\cdots+X_n$ with iid exponential increments,

$$
E[e^{-\alpha T_n}]
=\left(E[e^{-\alpha X_1}]\right)^n.
$$

For $X_1\sim \operatorname{Exponential}(\lambda)$,

$$
E[e^{-\alpha X_1}]=\frac{\lambda}{\alpha+\lambda}.
$$

Therefore

$$
E[C]=\beta\sum_{n=1}^\infty \left(\frac{\lambda}{\alpha+\lambda}\right)^n
=\beta\frac{\lambda}{\alpha}.
$$

For the chapter's numerical case:

- replacement cost $\beta=800$ dollars,
- mean lifetime $5000$ hours, so $\lambda=1/5000$ per hour,
- annual discount rate $24\%$, so $\alpha=0.24/(365\cdot24)$ per hour.

Then

$$
E[C]=800\cdot \frac{1/5000}{0.24/(365\cdot24)}=5840.
$$

In [ ]:
beta = 800
lam = 1/5000
alpha = 0.24/(365*24)
expected_cost = beta * lam / alpha
expected_cost

### Example 2.16 — every second vehicle

Suppose vehicles crossing a fixed point form a Poisson process $N$ with rate $\lambda$.

Let $U_1,U_2,\ldots$ be the successive interarrival times of the vehicles. Define

$$
V_1=U_1+U_2,
$$

$$
V_2=U_3+U_4,
$$

and so on. Then each $V_i$ has an Erlang-2 distribution.

If $M_t$ is the number of completed pairs by time $t$, then

$$
M_t=k
$$

means that $N_t$ is either $2k$ or $2k+1$. Hence

$$
P(M_t=k)=P(N_t=2k)+P(N_t=2k+1)
$$

so

$$
P(M_t=k)=e^{-\lambda t}\frac{(\lambda t)^{2k}}{(2k)!}
+e^{-\lambda t}\frac{(\lambda t)^{2k+1}}{(2k+1)!}.
$$

In [ ]:
lam = 3
T = 1
for k in range(5):
    print(k, poisson_pmf(2*k, lam*T) + poisson_pmf(2*k+1, lam*T))

## 13. Campbell-type formula for arrival sums

For any non-negative function $f$,

$$
E\left[\sum_{n=1}^{\infty} f(T_n)\right]
=\lambda\int_0^\infty f(t)\,dt.
$$

Proof idea:

Split time into small intervals. The sum over arrivals is approximately

$$
\sum_i f(t_i)\cdot \mathbf{1}\{\text{arrival in interval }i\}.
$$

Taking expectations gives approximately

$$
\sum_i f(t_i)\lambda\Delta t,
$$

which converges to the integral.

This is one of the most useful identities in applied Poisson-process calculations.

In [ ]:
# Check Campbell formula by simulation for f(t)=exp(-a t)
# The formula says E sum exp(-a T_n) = lambda / a.
lam = 2.0
alpha = 0.7
Tmax = 50.0
trials = 5000
values = []
for _ in range(trials):
    arr = poisson_path(lam, Tmax, rng)
    values.append(np.exp(-alpha * arr).sum())
np.mean(values), lam / alpha

## 14. Stopping times

A random time $T$ is a **stopping time** if, for each $t$, knowing the past up to time $t$ is enough to decide whether $T\le t$.

Examples:

- The time of the first arrival.
- The time of the sixth arrival.
- The first time a long enough gap appears in traffic.

For a Poisson process, a key property is:

> After a stopping time $T$, the future process $N_{T+t}-N_T$ behaves like a fresh Poisson process, independent of the past.

So for any stopping time $T$,

$$
P(N_{T+t}-N_T=k)=e^{-\lambda t}\frac{(\lambda t)^k}{k!}.
$$

### Example 2.19 — crossing traffic after a stopping time

Suppose vehicles pass a fixed point as a Poisson process with rate $\lambda=0.2$ per minute. A pedestrian chooses a stopping time $T$ based on observing traffic.

If the question is whether at least one vehicle arrives in the next $2$ minutes after $T$, then the stopping-time property gives

$$
P(N_{T+2}-N_T\ge1)=1-P(N_{T+2}-N_T=0)
=1-e^{-2\lambda}.
$$

With $\lambda=0.2$,

$$
P(N_{T+2}-N_T\ge1)=1-e^{-0.4}.
$$

In [ ]:
1 - math.exp(-0.4)

### Example 2.20 — bus inspector

Buses arrive at a stop according to a Poisson process with rate

$$
\lambda=0.2\quad\text{per minute}.
$$

An inspector arrives exactly one hour after the company opens, but each day the origin of time is chosen uniformly over the hour. The elapsed time since the previous origin is therefore $60$ minutes.

By stationarity, the number of buses that have arrived since the origin is

$$
\operatorname{Poisson}(0.2\cdot60)=\operatorname{Poisson}(12).
$$

So

$$
P(\text{exactly }k\text{ buses})=e^{-12}\frac{12^k}{k!}.
$$

In [ ]:
ks = np.arange(0, 30)
pmf = [poisson_pmf(k, 12) for k in ks]
plt.figure(figsize=(8, 3.5))
plt.bar(ks, pmf)
plt.xlabel("k")
plt.ylabel("probability")
plt.title("Number of buses since the random origin: Poisson(12)")
plt.grid(True, axis="y", alpha=0.3)
plt.show()

# Part III — Forward recurrence times

At a fixed time $t$, define:

- $U_t$: time since the last arrival before $t$.
- $V_t$: time from $t$ until the next arrival.

For a Poisson process, the future waiting time has the simple distribution

$$
P(V_t>s)=e^{-\lambda s}.
$$

So

$$
V_t\sim \operatorname{Exponential}(\lambda),
$$

and this distribution does not depend on $t$.

This is another form of memorylessness.

### Example 3.2 — the bus paradox intuition

Suppose buses arrive according to a Poisson process at rate

$$
\lambda=0.2\quad\text{per minute}.
$$

The average time between buses is

$$
\frac1\lambda=5\text{ minutes}.
$$

If you arrive at a random time, your expected waiting time until the next bus is also

$$
E[V_t]=5\text{ minutes}.
$$

But the expected time since the previous bus is also $5$ minutes. Therefore, the expected length of the interval containing your arrival time is

$$
E[U_t+V_t]=10\text{ minutes}.
$$

This does **not** contradict the average interarrival time of $5$ minutes. Random observation times are more likely to fall inside longer gaps.

In [ ]:
# Simulate the bus paradox: random observation times land in length-biased intervals.
lam = 0.2
Tmax = 100000
arr = poisson_path(lam, Tmax, rng)
obs = rng.uniform(1000, Tmax-1000, size=5000)
idx = np.searchsorted(arr, obs)
prev_arr = arr[idx-1]
next_arr = arr[idx]
elapsed = obs - prev_arr
wait = next_arr - obs
interval_len = next_arr - prev_arr
np.mean(wait), np.mean(elapsed), np.mean(interval_len), 1/lam

# Part IV — Superposition of Poisson processes

Let $L=\{L_t:t\ge0\}$ and $M=\{M_t:t\ge0\}$ be independent Poisson processes with rates $\lambda$ and $\mu$.

Define

$$
N_t=L_t+M_t.
$$

Then $N$ is a Poisson process with rate

$$
\lambda+\mu.
$$

Proof idea:

For each interval, the sum of two independent Poisson random variables is Poisson:

$$
\operatorname{Poisson}(\lambda t)+\operatorname{Poisson}(\mu t)
=\operatorname{Poisson}((\lambda+\mu)t).
$$

Independent increments are inherited from the two independent processes.

In [ ]:
lam, mu = 2.0, 3.0
T = 5.0
arr_L = poisson_path(lam, T, rng)
arr_M = poisson_path(mu, T, rng)
arr_N = np.sort(np.r_[arr_L, arr_M])
plot_counting_path(arr_N, T=T, title="Superposition: combined Poisson process")
len(arr_L), len(arr_M), len(arr_N)

### Example 4.3 — traffic from two directions

Suppose vehicles arriving from the north form a Poisson process with rate $\lambda$, and vehicles arriving from the south form an independent Poisson process with rate $\mu$.

The total arrival process at the intersection is a Poisson process with rate

$$
\lambda+\mu.
$$

The reason is exactly superposition: independent arrivals from multiple sources combine into one Poisson stream.

# Part V — Decomposition / thinning

Now go in the opposite direction.

Let $N$ be a Poisson process with rate $\lambda$. Suppose each arrival is independently classified as:

- type $1$ with probability $p_1$,
- type $2$ with probability $p_2$,
- $\ldots$
- type $m$ with probability $p_m$,

where $p_1+\cdots+p_m=1$.

Then the type-specific counting processes are independent Poisson processes with rates

$$
\lambda p_1,\ldots,\lambda p_m.
$$

This is called **thinning** or **decomposition**.

### Proof idea for thinning

Condition on $N_t=n$. Given $n$ arrivals, the type counts are multinomial:

$$
P(M_1=k_1,\ldots,M_m=k_m\mid N_t=n)
=\frac{n!}{k_1!\cdots k_m!}p_1^{k_1}\cdots p_m^{k_m}.
$$

Uncondition over

$$
N_t\sim \operatorname{Poisson}(\lambda t).
$$

The joint probability factors into a product of Poisson probabilities:

$$
M_i(t)\sim \operatorname{Poisson}(\lambda p_i t),
$$

and the $M_i$ are independent.

In [ ]:
# Simulate thinning of one Poisson process into three independent subprocesses.
lam = 10
T = 1.0
p = np.array([0.2, 0.5, 0.3])
trials = 10000
counts = np.zeros((trials, len(p)), dtype=int)
for i in range(trials):
    n = rng.poisson(lam*T)
    counts[i] = rng.multinomial(n, p)
counts.mean(axis=0), lam*T*p, counts.var(axis=0), lam*T*p

### Example 5.4 — left and right turns

Vehicles arrive at an intersection according to a Poisson process with rate $30$ vehicles per minute.

Each vehicle independently turns:

- left with probability $0.60$,
- right with probability $0.40$.

Then:

$$
\text{left turns} \sim \text{Poisson process with rate }30(0.60)=18,
$$

$$
\text{right turns} \sim \text{Poisson process with rate }30(0.40)=12.
$$

The left-turn and right-turn processes are independent.

### Example 5.5 — restaurant arrivals by party size

Vehicles arrive at a roadside restaurant according to a Poisson process with rate

$$
\lambda=20\quad\text{per hour}.
$$

The number of passengers in a vehicle has probabilities:

$$
P(1)=0.30,
$$

$$
P(2)=0.30,
$$

$$
P(3)=0.20,
$$

$$
P(4)=0.10,
$$

$$
P(5)=0.10.
$$

By thinning, the streams of cars with $1,2,3,4,5$ passengers are independent Poisson processes with rates:

$$
20(0.30),\;20(0.30),\;20(0.20),\;20(0.10),\;20(0.10).
$$

The expected number of arriving passengers per hour is

$$
20\cdot E[\text{party size}]
=20(1\cdot0.30+2\cdot0.30+3\cdot0.20+4\cdot0.10+5\cdot0.10)=48.
$$

In [ ]:
lam = 20
sizes = np.array([1,2,3,4,5])
probs = np.array([0.30,0.30,0.20,0.10,0.10])
lam * np.dot(sizes, probs)

# Part VI — Compound Poisson processes

A compound Poisson process allows jumps of arbitrary random sizes.

Let:

- $N_t$ be a Poisson process with rate $\lambda$,
- $X_1,X_2,\ldots$ be iid jump sizes,
- the $X_i$ be independent of $N$.

Define

$$
Z_t=\sum_{i=1}^{N_t}X_i.
$$

Then $Z=\{Z_t:t\ge0\}$ is a **compound Poisson process**.

Thinking model:

- $N_t$ tells how many events happened.
- $X_i$ tells the size or cost of the $i$th event.
- $Z_t$ totals the sizes up to time $t$.

In [ ]:
# Simulate a compound Poisson process with normally distributed jump sizes.
rate = 2.0
T = 5.0
arr = poisson_path(rate, T, rng)
jumps = rng.normal(loc=1.0, scale=0.8, size=len(arr))
levels = np.cumsum(jumps)

xs = [0.0]
ys = [0.0]
level = 0.0
for t_arr, jump in zip(arr, jumps):
    xs.extend([t_arr, t_arr])
    ys.extend([level, level + jump])
    level += jump
xs.append(T)
ys.append(level)

plt.figure(figsize=(8, 3.5))
plt.step(xs, ys, where="post")
plt.scatter(arr, levels, zorder=3)
plt.xlabel("t")
plt.ylabel(r"$Z_t$")
plt.title("Compound Poisson sample path with random jump sizes")
plt.grid(True, alpha=0.3)
plt.show()

## 15. Characterization of compound Poisson processes

A process $Z$ is compound Poisson exactly when:

1. its jump times form a Poisson process, and
2. its successive jump magnitudes are iid and independent of the jump times.

This is the compound analog of “arrival times + marks.”

### Example 6.3 — total store sales

Customers arrive according to a Poisson process $N$.

Let $X_i$ be the amount of money spent by the $i$th customer. Suppose the $X_i$ are iid and independent of the arrival process.

The total sales by time $t$ are

$$
Z_t=X_1+\cdots+X_{N_t}.
$$

Then $Z$ is a compound Poisson process.

### Example 6.4 — cumulative repair cost

Failures occur at Poisson arrival times. Each failure has a random repair cost $X_i$, iid and independent of the failure times.

The cumulative repair cost is

$$
Z_t=\sum_{i=1}^{N_t}X_i.
$$

So cumulative repair cost is a compound Poisson process.

## 16. Expectation of a compound Poisson process

If

$$
E[X_1]=\mu,
$$

then conditioning on $N_t$ gives

$$
E[Z_t\mid N_t=n]=n\mu.
$$

Taking expectations:

$$
E[Z_t]=E[N_t]\mu=\lambda t\mu.
$$

This is a clean example of the rule:

> expected total = expected count $\times$ expected size.

## 17. Laplace transform of a compound Poisson process

Suppose the jumps are non-negative and

$$
f(\alpha)=E[e^{-\alpha X_1}].
$$

Then

$$
E[e^{-\alpha Z_t}]=\exp\{-\lambda t(1-f(\alpha))\}.
$$

Proof:

Condition on $N_t=n$:

$$
E[e^{-\alpha Z_t}\mid N_t=n]
=E[e^{-\alpha(X_1+\cdots+X_n)}]
=f(\alpha)^n.
$$

Then

$$
E[e^{-\alpha Z_t}]=E[f(\alpha)^{N_t}].
$$

Since $N_t\sim\operatorname{Poisson}(\lambda t)$,

$$
E[r^{N_t}]=e^{-\lambda t(1-r)}.
$$

Set $r=f(\alpha)$.

In [ ]:
# Simulation check for Laplace transform.
rate = 3.0
T = 2.0
alpha = 0.5
# jump sizes exponential with rate beta_j, so f(alpha)=beta_j/(beta_j+alpha)
beta_j = 4.0
trials = 20000
vals = []
for _ in range(trials):
    n = rng.poisson(rate*T)
    jumps = rng.exponential(1/beta_j, size=n)
    vals.append(math.exp(-alpha * jumps.sum()))
empirical = np.mean(vals)
f = beta_j / (beta_j + alpha)
theory = math.exp(-rate*T*(1-f))
empirical, theory

## 18. Decomposing compound Poisson by jump size

If jumps take values in a countable set $E=\{a,b,\ldots\}$, then the arrivals with jump size $a$ form a Poisson process with rate

$$
\lambda P(X=a),
$$

and similarly for $b,\ldots$. These subprocesses are independent.

Thus

$$
Z_t=aN_t^a+bN_t^b+\cdots.
$$

This is thinning applied to the jump-size mark.

# Part VII — Non-stationary Poisson processes

In many real arrival systems, the arrival rate changes over time.

A **possibly non-stationary Poisson process** keeps:

1. unit jumps,
2. independent increments,

but drops stationary increments.

Define the expectation function

$$
a(t)=E[N_t].
$$

Then $a(t)$ is non-decreasing and right-continuous.

For $0\le s<t$,

$$
E[N_t-N_s]=a(t)-a(s).
$$

## 19. Continuous expectation function

When $a$ is continuous, the non-stationary Poisson process satisfies

$$
P(N_t-N_s=k)=e^{-(a(t)-a(s))}\frac{(a(t)-a(s))^k}{k!}.
$$

So increments are still Poisson, but the mean is not $\lambda(t-s)$; it is

$$
a(t)-a(s).
$$

If $a$ is differentiable, then

$$
\lambda(t)=a'(t)
$$

is the instantaneous rate.

In [ ]:
# Non-stationary Poisson example with intensity lambda(t)=2+sin(t), so a(t)=2t+1-cos(t).
def a(t):
    return 2*t + 1 - np.cos(t)

T = 10
# simulate by time-change: generate stationary Poisson arrivals in a-time, then invert a(t) numerically
S = poisson_path(1.0, float(a(T)), rng)
grid = np.linspace(0, T, 5000)
agrid = a(grid)
arrivals_t = np.interp(S, agrid, grid)
plot_counting_path(arrivals_t, T=T, title="Non-stationary Poisson process via time change")

## 20. Time-change representation

If $M$ is a standard Poisson process with rate $1$, and $a(t)$ is continuous and non-decreasing, then

$$
N_t=M_{a(t)}
$$

is a non-stationary Poisson process with expectation function $a$.

Conversely, if $N$ is a non-stationary Poisson process with continuous $a$, its arrival times can be generated by:

1. Generate iid exponential$(1)$ variables $X_1,X_2,\ldots$.
2. Form $S_n=X_1+\cdots+X_n$.
3. Solve

$$
a(T_n)=S_n.
$$

That is,

$$
T_n=a^{-1}(S_n).
$$

## 21. Discontinuities in $a(t)$: scheduled arrivals

If $a$ has a jump at time $t$,

$$
a(t)-a(t-)=\alpha,
$$

then there is a scheduled possible arrival exactly at time $t$ with probability $\alpha$:

$$
P(N_t-N_{t-}=1)=\alpha.
$$

So a general non-stationary Poisson process can be decomposed into:

1. a continuous-rate stream, and
2. fixed-time scheduled possible arrivals.

### Example 7.17 — barber shop with appointments and walk-ins

Customers at a barber shop arrive from two streams:

1. Appointments at fixed times: 12:00, 12:20, 1:00, 3:40, 4:20, 4:40.
2. Walk-ins with a time-varying rate:
   - rate $1$ per hour during the first 3 hours,
   - rate $0.4$ per hour during the next 2 hours,
   - rate $0.2$ per hour during the last hour.

Each appointment is kept with probability

$$
\alpha=\frac23.
$$

During the first 4 hours, there are 4 scheduled appointments. Thus the number of kept appointments has distribution

$$
\operatorname{Binomial}\left(4,\frac23\right).
$$

The expected number of walk-ins in the first 4 hours is

$$
3\cdot1+1\cdot0.4=3.4.
$$

So walk-ins in the first 4 hours have distribution

$$
\operatorname{Poisson}(3.4).
$$

The total number of customers by hour 4 is the independent sum of these two variables.

For example, the probability of exactly 8 customers by hour 4 is

$$
\sum_{k=0}^4
\binom{4}{k}\left(\frac23\right)^k\left(\frac13\right)^{4-k}
\cdot e^{-3.4}\frac{3.4^{8-k}}{(8-k)!}.
$$

In [ ]:
prob = 0.0
for k in range(5):
    binom = math.comb(4, k) * (2/3)**k * (1/3)**(4-k)
    pois = poisson_pmf(8-k, 3.4)
    prob += binom * pois
prob

# Summary map

| Concept | Main formula | Mental model |
|---|---:|---|
| Poisson count | $P(N_t=k)=e^{-\lambda t}(\lambda t)^k/k!$ | arrivals in length $t$ |
| Independent increments | counts on disjoint intervals independent | future independent of past |
| Arrival times | $T_n=X_1+\cdots+X_n$ | cumulative waiting times |
| Interarrival time | $X_i\sim \operatorname{Exp}(\lambda)$ | memoryless waiting |
| Erlang time | $T_n\sim \operatorname{Gamma}(n,\lambda)$ | time of $n$th arrival |
| Superposition | rates add | merge independent streams |
| Thinning | rates multiply by probabilities | classify arrivals independently |
| Compound Poisson | $Z_t=\sum_{i=1}^{N_t}X_i$ | random total over random count |
| Non-stationary | mean increment $a(t)-a(s)$ | rate changes over time |

# Exercises from the chapter — how to approach them

The exercises use the same toolkit repeatedly.

## Exercise pattern 1: count probabilities

For questions like $P(N_t=k)$ or joint events involving $N_{t_1},N_{t_2},\ldots$, rewrite everything as independent increments.

Example template:

$$
P(N_5=9,N_{10}=13)
=P(N_5=9)P(N_{10}-N_5=4).
$$

## Exercise pattern 2: conditional count probabilities

For $P(N_s=a\mid N_t=b)$ with $s<t$, condition on total arrivals. Then the count in the subinterval is binomial:

$$
N_s\mid N_t=b\sim \operatorname{Binomial}\left(b,\frac{s}{t}\right).
$$

## Exercise pattern 3: random stopping or inspection times

Condition on the random time $T$:

$$
E[N_T\mid T]=\lambda T,
$$

and

$$
\operatorname{Var}(N_T)=E[\lambda T]+\operatorname{Var}(\lambda T).
$$

## Exercise pattern 4: thinning

If arrivals are classified independently, each class is Poisson with rate $\lambda p_i$.

## Exercise pattern 5: compound sums

For totals

$$
Z_t=\sum_{i=1}^{N_t}X_i,
$$

use conditioning on $N_t$:

$$
E[Z_t]=\lambda t E[X],
$$

and, when needed,

$$
\operatorname{Var}(Z_t)=\lambda t E[X^2].
$$

The last formula follows from the law of total variance:

$$
\operatorname{Var}(Z_t)=E[N_t\operatorname{Var}(X)]+\operatorname{Var}(N_tE[X])
=\lambda t\operatorname{Var}(X)+\lambda t(E[X])^2.
$$